## __Aprendizaje no supervisado__

__Profesor__: Anthony D. Cho

__Ayudante__: Luis Oliveros

__Asunto__: Medidas de evaluación

***

### Librerias

In [1]:
from pandas import read_csv, DataFrame
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

## Metodo de estandarizacion
from sklearn.preprocessing import StandardScaler

## Metodos de clustering
from sklearn.cluster import KMeans, DBSCAN
import scipy.cluster.hierarchy as shc

## Otros metodos
from sklearn.neighbors import NearestNeighbors

## Metricas de performance en clusters
from sklearn.metrics import silhouette_score, davies_bouldin_score

__Informacion de:__

* [Metrica de Silhouette](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html#sklearn.metrics.silhouette_score): esta entre [-1, 1]. Cercano a 1 implica mejor modelo. [Detalles ...](https://es.wikipedia.org/wiki/Silhouette_(clustering))

* [Metrica de Davies-Bouldin](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.davies_bouldin_score.html#sklearn.metrics.davies_bouldin_score): Minimo valor es 0, cercano a 0 implica mejor modelo. [Detalles ...](https://en.wikipedia.org/wiki/Davies%E2%80%93Bouldin_index)

### Data
__Fuente__: [UCI Repository: Seed](https://archive.ics.uci.edu/ml/datasets/seeds)

<center>
    <img src='https://upload.wikimedia.org/wikipedia/commons/a/a3/Vehn%C3%A4pelto_6.jpg' width=800>
</center>

In [ ]:
## Cargar los datos
data = read_csv('https://raw.githubusercontent.com/adoc-box/Datasets/main/seeds.txt', header=None, sep='\t')
data.columns = ['area', 'perimeter', 'compactness', 'length of kernel', 'width of kernel', 
                'asymmetry coefficient', 'length of kernel groove', 'wheatType']
data.drop(columns=['wheatType'], inplace=True)  ## <-- obviamos las etiquetas

## Mostrar los primeros 5 registros
data.head(5)

In [3]:
## Escalamos la data
scaler = StandardScaler().fit(data)
data_scaled = scaler.transform(data)

data_scaled = DataFrame(data_scaled, columns=data.columns)

## K-Means

#### Busqueda del mejor número de clusters

In [ ]:
## Almacenamiento de informacion de rendimientos
rendimiento = {'k': [], 'distancia': [], 'silhouette': [], 'Davies-Bouldin': []}

## Encontremos el mejor numero de clusters
distancia_intraCluster = []

max_clusters = 21
K_values = list(range(2, max_clusters))
for k in tqdm(K_values):
    
    ## Instancia del modelo
    model = KMeans(n_clusters=k, n_init=10)
    
    ## Ajuste del modelo
    model.fit(data_scaled)
    
    ## Prediccion
    prediccion = model.predict(data_scaled)
    
    ## Almacenar la distancia intra-cluster del modelo ajustado
    rendimiento['distancia'].append( model.inertia_ )
    
    ## Almacenar metrica de desempeño
    rendimiento['silhouette'].append( silhouette_score(X=data_scaled, 
                                                       labels=prediccion, 
                                                       metric='euclidean') )
    rendimiento['Davies-Bouldin'].append( davies_bouldin_score(X=data_scaled, 
                                                               labels=prediccion) )
    rendimiento['k'].append(k)

## Dataframe del rendimiento de los resultados
rendimiento = DataFrame(rendimiento)
rendimiento

In [ ]:
## Grafica de codo. 
plt.figure(figsize=(16, 5))
sns.lineplot(data=rendimiento, x='k', y='distancia')
plt.title('Elbow curve')
plt.xlabel('Numero de cluster')
plt.ylabel('SSE')
plt.tight_layout()
plt.show()

In [ ]:
## Grafica de las metricas
plt.figure(figsize=(16, 5))
plt.plot(rendimiento['k'], rendimiento['silhouette'], 'ro-', label='silhoutte')
plt.plot(rendimiento['k'], rendimiento['Davies-Bouldin'], 'bo-', label='Davies-Bouldin')
plt.xlabel('k'); plt.ylabel('metric-score'); plt.legend(); plt.grid()
plt.tight_layout(); plt.show()

__¿Cual es el mejor valor de k?__ <br/>
Por la grafica de codo, podemos considerar k=3.

#### Mejor modelo

In [ ]:
## Instancia del modelo
model = KMeans(n_clusters=3, n_init=100, random_state=9001)

## Ajuste del modelo
model.fit(data_scaled)

In [ ]:
## Valor de distancia intra-cluster obtenida con el mejor modelo
model.inertia_

In [ ]:
## Imprimir lista de muestra-a-cluster
model.labels_

In [ ]:
## Graficamos por par de variables

report_model = data_scaled.copy()
report_model['Class'] = model.labels_.astype(str)

sns.pairplot(data=report_model, hue='Class', corner=True)

In [21]:
## Se reestructura los datos para efecto de visualizacion
data_cluster_melt = report_model.melt(id_vars='Class')

In [ ]:
## Graficado de la distribución de cada variable por grupo
ncol = 4
nrow = report_model.shape[1] // ncol +1
nrow = nrow if nrow>0 else 1
print('nrow: {} - ncol: {}'.format(nrow, ncol))

plt.figure(figsize=(16, 4*nrow))
for i, var in enumerate(data.columns):

    plt.subplot(nrow, ncol, i+1)
    temp = data_cluster_melt.query(f'variable == "{var}"')
    sns.pointplot(
        data=temp, x="Class", y="value",
        errorbar=('pi', 100), capsize=.4, join=False, color="blue",
    )
    plt.title(f'Var: {var}')
plt.tight_layout()
plt.show()

## Density-Based Spatial Clustering of Applications with Noise (DBSCAN)

#### Busqueda del valor apropiado para eps

In [ ]:
## Numero de vecinos
n_vecinos = 10

## Instancia del modelo
neighbors_model = NearestNeighbors(n_neighbors=n_vecinos)

## Ajuste del modelo
neighbors_model = neighbors_model.fit(data_scaled)

## Computar de los n vecinos más cercano y los indices de las muestras vecinos
distancias, indices = neighbors_model.kneighbors(data_scaled)

## Reservamos la distancia maxima del vecino cercano para cada punto y los ordenamos
distancias = distancias.max(axis=1)
distancias.sort()

## Agrupemos la informacion en un dataframe
reporte = DataFrame({'i-esimo punto': range(1, len(distancias)+1),
                    'distancia': distancias})
reporte

In [ ]:
## Numero de puntos
N = len(reporte)

plt.figure(figsize=(13, 5))
sns.lineplot(data=reporte, x='i-esimo punto', y='distancia')
plt.xlabel('i-ésimo punto')
plt.ylabel('Distancia')
plt.title('Distancia al {}° vecino más cercano de cada punto'.format(n_vecinos))
plt.tight_layout()
plt.grid()
plt.show()

#### Mejor modelo

Supongamos que toleramos a una distancia de 0.9.

In [ ]:
## Instancia del modelo
model = DBSCAN(eps=0.9, min_samples=n_vecinos, n_jobs=-1)

## Ajuste del modelo
model.fit(data_scaled)

In [ ]:
## Graficamos por par de variables
report_model = data_scaled.copy()
report_model['Class'] = model.labels_.astype(str)

sns.pairplot(data=report_model, hue='Class', corner=True)

In [ ]:
## Se reestructura los datos para efecto de visualizacion
data_cluster_melt = report_model.melt(id_vars='Class')

## Graficado de la distribución de cada variable por grupo
ncol = 4
nrow = report_model.shape[1] // ncol +1
nrow = nrow if nrow>0 else 1
print('nrow: {} - ncol: {}'.format(nrow, ncol))

plt.figure(figsize=(16, 4*nrow))
for i, var in enumerate(data.columns):

    plt.subplot(nrow, ncol, i+1)
    temp = data_cluster_melt.query(f'variable == "{var}"')
    sns.pointplot(
        data=temp, x="Class", y="value",
        errorbar=('pi', 100), capsize=.4, join=False, color="blue",
    )
    plt.title(f'Var: {var}')
plt.tight_layout()
plt.show()